# Tutorial: Formal Verification of Safety-Critical Surrogate Models

This tutorial demonstrates the end-to-end pipeline for ensuring the reliability of a Neural Network (NN) used in aeronautics. We focus on **Braking Distance Estimation**, a function where a prediction error isn't just a metric—it's a safety risk.

### Learning Objectives

1. **Train a Surrogate Model**: Learn how to approximate a complex physical simulation (Cessna landing) using a Neural Network.
2. **Define Formal Properties**: Translate physical safety requirements (e.g., "Always overestimate braking distance") into mathematical constraints.
3. **Formal Verification**: Use the `airobas` library to provide mathematical guarantees that the NN respects these properties across the entire input space.


## Cell 2: Why Verification Matters (Markdown)

In safety-critical systems, **Statistical Accuracy** (e.g., low MSE) is insufficient. We require **Formal Guarantees**.

* **Reliability**: NNs are notoriously non-linear; small input changes can lead to unexpected spikes in output.
* **Safety vs. Cost**: In landing scenarios, under-predicting braking distance leads to runway overruns (Safety Risk), while over-predicting leads to unnecessary go-arounds (Operational Cost).
* **The Goal**: We want to prove that our model is "Safe by Design" by ensuring it never under-predicts the reference simulation.


In [ ]:
# Install dependencies if on Colab
on_colab = "google.colab" in str(get_ipython())
if on_colab:
    !{sys.executable} -m pip install -U pip
    !{sys.executable} -m pip install git+https://github.com/ducoffeM/AIBT_introduction_2_verification
    from IPython.display import clear_output
    clear_output()

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import keras
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from keras.layers import Dense
from keras.models import Sequential

# Set Backend to Torch (Required for certain verification engines)
os.environ["KERAS_BACKEND"] = "torch"

import cesna
from cesna import generate_cesna_dataset
from cesna.cesna_simulation import cesna_landing

We define our operational design domain (ODD):

* **Temperature**: $0^\circ\text{C}$ to $40^\circ\text{C}$
* **Altitude**: $100\text{ ft}$ to $4000\text{ ft}$

In [ ]:
MIN_temp, MAX_temp = 0, 40
MIN_alt, MAX_alt = 100, 4000

# Generate Datasets
X_train, y_train = generate_cesna_dataset(10000, MIN=[MIN_temp, MIN_alt], MAX=[MAX_temp, MAX_alt])
X_valid, y_valid = generate_cesna_dataset(1000, MIN=[MIN_temp, MIN_alt], MAX=[MAX_temp, MAX_alt])
X_test, y_test = generate_cesna_dataset(5000, MIN=[MIN_temp, MIN_alt], MAX=[MAX_temp, MAX_alt])

# Feature Scaling (Crucial for NN convergence and Verification efficiency)
MEAN_x, STD_x = np.mean(X_train, 0)[None], np.std(X_train, 0)[None]
MEAN_y, STD_y = np.mean(y_train, 0), np.std(y_train, 0)

X_train_n = (X_train - MEAN_x) / STD_x
y_train_n = (y_train - MEAN_y) / STD_y
X_test_n = (X_test - MEAN_x) / STD_x
y_test_n = (y_test - MEAN_y) / STD_y

## Training a "Biased" Model (Code)

To favor safety, we intentionally add a slight bias during training to encourage the model to overestimate.


In [ ]:
model = Sequential([
    Dense(20, input_dim=2, activation="relu"),
    Dense(20, activation="relu"),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse")

# Training with a 10% safety bias to encourage over-estimation
model.fit(X_train_n, y_train_n + 0.1 * np.abs(y_train_n), 
          epochs=10, batch_size=32, validation_split=0.1, verbose=0)

print("Model trained. Evaluating on test set...")
model.evaluate(X_test_n, y_test_n)

### The Monotonicity Assumption

In aviation physics, braking distance $f$ is monotonic:


$$if \ x_1 \leq x_2 \implies f(x_1) \leq f(x_2)$$

### The Safety Property

We divide the input space into a grid of cells $C_i$. For each cell, we define the **Worst-Case Reference Value** $y_{ref, i}$ as the value at the top-right corner (highest temperature, highest altitude).

**Formal Statement:**


$$\forall x \in C_i, \quad NN(x) \geq y_{ref, i}$$


If this holds, the model never underestimates the distance within that cell.

In [ ]:
from aibt_fm import create_grid_cells, verif_with_airobas
from airobas.blocks_hub.decomon_block import DecomonBlock
from airobas.blocks_hub.adv_block import CleverhansAdvBlock

# 1. Create the grid
grid = create_grid_cells(bottom_left=[MIN_temp, MIN_alt], top_right=[MAX_temp, MAX_alt], granularity=11)

# 2. Define normalization wrappers for Airobas
def get_box(cells):
    # Transforms physical cells into normalized NN input space
    return (cells - MEAN_x) / STD_x

def get_ground_dist(cells):
    # Computes the normalized reference value (top-right corner of cell)
    y_raw = np.array([cesna_landing(cell[1, 0], cell[1, 1]) for cell in cells])
    return (y_raw[:, None] - MEAN_y) / STD_y

In [ ]:
## Cell 8: Running the Verification Pipeline (Code)

We use two types of verification blocks:

1. **Adversarial Block**: Tries to find a counter-example (Proof of Unsafety).
2. **Decomon Block**: Uses CROWN to provide a mathematical proof of safety.

In [ ]:
 Sequential Verification: First find easy bugs (Adv), then prove safety (Decomon)
blocks = [
    (CleverhansAdvBlock, {"index_target": 0, "attack_up": False, "fgs": True}),
    (DecomonBlock, {})
]

print("Starting verification...")
is_verified, is_not_verified, is_unsafe = verif_with_airobas(model, grid, get_box, get_ground_dist, blocks)

print(f"Verified Cells: {len(is_verified)}")
print(f"Unsafe Cells (Counter-example found): {len(is_unsafe)}")
print(f"Undecided Cells: {len(is_not_verified)}")


In [ ]:
## Cell 9: Result Visualization

In [ ]:
def plot_verif_results(grid, verified, unsafe, undecided):
    plt.figure(figsize=(10, 7))
    ax = plt.gca()
    
    for idx, (res_list, color, label) in enumerate([
        (verified, 'green', 'Safe'),
        (unsafe, 'red', 'Unsafe'),
        (undecided, 'orange', 'Undecided')
    ]):
        for cell_idx in res_list:
            cell = grid[cell_idx]
            rect = patches.Rectangle(cell[0], cell[1,0]-cell[0,0], cell[1,1]-cell[0,1], 
                                     color=color, alpha=0.3)
            ax.add_patch(rect)
            if cell_idx == res_list[0]: rect.set_label(label) # Label first for legend

    plt.xlabel("Temperature (°C)")
    plt.ylabel("Altitude (ft)")
    plt.title("Formal Verification Map of Braking Surrogate")
    plt.legend()
    plt.show()

plot_verif_results(grid, is_verified, is_unsafe, is_not_verified)

### Key Takeaways

* **Adversarial attacks** are your first line of defense; if an attack finds a violation, you don't need expensive formal methods to know the model is unsafe.
* **Incomplete methods** (like those in `airobas` and `Decomon`) are the industry standard for neural networks because they provide safety guarantees while remaining computationally feasible for real-world architectures.
* **"Unknown"** does not mean the model is unsafe; it means the verification method was not precise enough to prove safety. This is often solved by **refining** the input grid into smaller cells.



### Verification Method Comparison

This table summarizes the different families of verification techniques. It is essential to understand these trade-offs when selecting a tool for safety-critical AI.

| Method Category | Example Tool / Logic | Guarantee Type | Computational Cost | Result Output |
| --- | --- | --- | --- | --- |
| **Adversarial (Empirical)** | `CleverhansAdvBlock` (FGSM, PGD) | **No Guarantee.** Useful for finding bugs and counter-examples. | **Low**: Very fast to compute. | "Unsafe" (if violation found) or "Unknown" |
| **Incomplete (Formal)** | `DecomonBlock` (IBP, CROWN, Box) | **Sound.** If the tool says "Safe," it is mathematically proven. | **Medium**: Scales well to larger networks. | "Safe" or "Unknown" (due to over-approximation) |
| **Complete (Formal)** | MILP / SMT Solvers (e.g., Marabou, nnenum) | **Exact.** Always provides a definitive "Yes" or "No." | **High**: Exponential complexity; limited to smaller models. | "Safe" or "Unsafe" |

## Challenges Implementation

### Challenge 1: The Safety-Accuracy Trade-off

In this exercise, we compare a "Safety-Biased" model (trained with an offset) against a "Standard" model.

```python
# 1. Standard Model (No Bias)
model_std = Sequential([Dense(20, input_dim=2, activation="relu"), Dense(20, activation="relu"), Dense(1)])
model_std.compile(optimizer="adam", loss="mse")
model_std.fit(X_train_n, y_train_n, epochs=10, verbose=0)

# 2. Re-run verification for the Standard Model
print("Verifying Standard Model...")
v_std, n_std, u_std = verif_with_airobas(model_std, grid, get_box, get_ground_dist, blocks)

# 3. Compare Results
print(f"Safety-Biased Model Verified: {len(is_verified)} cells")
print(f"Standard Model Verified: {len(v_std)} cells")

```

**Key Takeaway:** Students will see that the Standard model has a better MSE but fails the safety property in many more regions. This illustrates that **optimization objectives** in ML must be aligned with **system requirements** in engineering.

---

### Challenge 2: Precision through Refinement

If a cell is "Undecided" (Orange), it means the over-approximation of the NN bounds was too "loose" to prove safety, even if the model is actually safe.

```python
# Increase granularity from 11 to 20
fine_grid = create_grid_cells(bottom_left=[MIN_temp, MIN_alt], 
                              top_right=[MAX_temp, MAX_alt], 
                              granularity=20)

print(f"Analyzing {len(fine_grid)} smaller cells...")
v_fine, n_fine, u_fine = verif_with_airobas(model, fine_grid, get_box, get_ground_dist, blocks)

```

**Key Takeaway:** Smaller cells provide tighter bounds for methods like **CROWN**.

---

## Summary & Conclusion

### Summary of the Pipeline

In this tutorial, we transitioned from a standard data-driven approach to a verified engineering approach:

* **Data Acquisition**: Defined a flight envelope (Temperature/Altitude) and generated ground-truth landing data.
* **Property Formulation**: Translated the physical need for over-estimation into a mathematical inequality: $NN(x) \geq y_{ref}$.
* **Verification**: Leveraged **Adversarial Attacks** to find immediate violations and **Sound Abstract Interpretation** (Decomon) to provide mathematical proofs of safety.

### Conclusion

Formal verification is becoming a pillar of **Trustworthy AI**. While traditional testing (validation/test sets) samples discrete points, formal methods provide guarantees over **continuous intervals**. For a Master student, the primary lesson is that the "best" model is not always the one with the highest accuracy, but the one whose behavior can be rigorously bounded and proven safe within its operational environment.
